<a href="https://colab.research.google.com/github/jmduea/orbit_wars/blob/main/notebooks/colab_compute.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [code]
#!/usr/bin/env python3
"""Self-contained Kaggle bootstrap for Orbit Wars population workers."""

from __future__ import annotations

import base64
import hashlib
import io
import json
import runpy
import sys
import tarfile
from pathlib import Path

ORBIT_WARS_PACKAGE_BOOTSTRAP = 'embedded-payload-v6'
_PAYLOAD_SHA256 = '8687823a49709480c47cc8cf525f1f48d11baa2a5934b01f3cbf4aceae1ded92'
_PAYLOAD_MANIFEST = json.loads('{"file_count": 198, "has_kaggle_jax": true, "has_worker": true, "mode": "embedded-payload-v6", "sample": ["src/__init__.py", "src/artifacts/__init__.py", "src/artifacts/checkpoint_compat.py", "src/artifacts/checkpoint_retention.py", "src/artifacts/pipeline.py", "src/artifacts/promotion.py", "src/artifacts/replay.py", "src/artifacts/run_paths.py", "src/artifacts/tournament/__init__.py", "src/artifacts/tournament/eval.py", "src/artifacts/tournament/promotion.py", "src/artifacts/tournament/ranking.py", "src/artifacts/tournament/resolve.py", "src/artifacts/tournament/runner.py", "src/artifacts/tournament/types.py", "src/artifacts/tournament/worker.py", "src/artifacts/worker_env.py", "src/cli/__init__.py", "src/cli/eval.py", "src/cli/kaggle_runner.py", "src/cli/train_hosts.py", "src/config/__init__.py", "src/config/rollout_allocation.py", "src/config/runtime.py", "src/config/schema.py"]}')
_PAYLOAD_B64 = (
    "..."
)


def _extract_payload(root: Path, payload: bytes) -> None:
    with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
        try:
            archive.extractall(root, filter="data")
        except TypeError:
            archive.extractall(root)


def main() -> None:
    root = Path.cwd()
    payload = base64.b64decode(_PAYLOAD_B64)
    actual_sha256 = hashlib.sha256(payload).hexdigest()
    print("ORBIT_WARS_PACKAGE_BOOTSTRAP=" + ORBIT_WARS_PACKAGE_BOOTSTRAP, flush=True)
    print("ORBIT_WARS_PACKAGE_PAYLOAD_SHA256=" + actual_sha256, flush=True)
    print("ORBIT_WARS_PACKAGE_PAYLOAD_MANIFEST=" + json.dumps(_PAYLOAD_MANIFEST, sort_keys=True), flush=True)
    if actual_sha256 != _PAYLOAD_SHA256:
        raise SystemExit(
            "embedded payload sha256 mismatch: " + actual_sha256 + " != " + _PAYLOAD_SHA256
        )
    _extract_payload(root, payload)
    worker = root / "scripts" / "kaggle_worker_entry.py"
    if not worker.exists():
        top_level = sorted(path.name for path in root.iterdir())
        raise SystemExit(
            "embedded payload extraction did not create scripts/kaggle_worker_entry.py; "
            + "cwd=" + str(root)
            + " top_level=" + ",".join(top_level)
        )
    root_text = str(root)
    if root_text not in sys.path:
        sys.path.insert(0, root_text)
    runpy.run_path(str(worker), run_name="__main__")


if __name__ == "__main__":
    main()
